Ask a traveller to describe their dream Sydney stay, and they'll probably start with the beach view or the walk to the harbour — not the host's response rate or how many other listings the host manages. But this dataset proves that much more shapes a listing's nightly price than bedrooms and a nice photo.

With 34 variables describing every Airbnb listing in Sydney — 20,573 of them in the June 2026 snapshot — the problem is to understand what drives the price of a stay.

<b>Acknowledgments</b> <br />
The data comes from Inside Airbnb (insideairbnb.com), an independent open-data project that publishes quarterly snapshots of Airbnb's public listings, used here under its Creative Commons Attribution 4.0 licence. This is the Sydney snapshot of 16 June 2026 — the same listings you could find on the site that week.

<h1>Exploratory Data Analysis (EDA)</h1>
Exploratory Data Analysis (EDA) is an approach used in data science to summarize and visually analyze the characteristics of a data set. It helps to answer important questions related to using machine learning algorithms in a project, selecting the most suitable algorithms, defining feature variables for machine learning, and finding insights that were not evident to stakeholders.

EDA involves multiple exploratory techniques, including :
<ul>
<li>univariate</li>
<li>bivariate </li>
<li>multivariate visualization</li>
<li>dimensionality reduction</li>
</ul>

By validating assumptions and identifying patterns, EDA ensures that the data set is ready to be used in a machine learning algorithm. Skipping or doing a shallow work on EDA can lead to inaccurate or inefficient models, choosing the wrong variables, and other negative outcomes. Data profiling is an important aspect of EDA, as it provides a solid understanding of the data set and allows for querying and visualizing the data in various ways.

In [ ]:
# pandas      -> loading and working with the data table
# numpy       -> number types (used when selecting numeric columns)
# matplotlib  -> plotting
# seaborn     -> the heatmaps and pair plot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# read the Sydney Airbnb listings dataset into a DataFrame called df
df = pd.read_csv('data/sydney_airbnb.csv')

In [ ]:
# display the table - 20,573 listings, 34 columns
df

A good first step is a basic overview of the data — a quick, informal look at its shape, ranges and averages. The describe() method does this in one line.

In [ ]:
# summary statistics for every numeric column:
# count, mean, spread (std), smallest, largest, and the quartiles
df.describe()

In [ ]:
# just the first 5 rows - a quick look at what the columns hold
df.head()

In [ ]:
# (rows, columns) - how big is this dataset?
df.shape

<h1>Missing values</h1>

Unlike the first lab's dataset, this one has real gaps. Counting them tells us two different stories: some values are missing because they cannot exist yet (a listing with no reviews has no review scores), while others were simply never recorded (bathrooms, beds) — an important distinction before deciding how to treat them.

In [ ]:
# count the missing values in every column,
# then keep only the columns that have any, biggest first
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

<h1> Numerical Columns </h1>

In [ ]:
# select only the columns that hold numbers (any numeric type)
numeric_features = df.select_dtypes(include=[np.number])

numeric_features.columns

<h1> Categorical Columns </h1>

In [ ]:
# select only the columns that hold text - the categorical features
categorical_features = df.select_dtypes(include=['object'])

categorical_features.columns

In [ ]:
# correlation between every pair of numeric columns, as one big heatmap
# vmax=0.8 caps the colour scale so the strongest links stand out clearly
plt.figure(figsize=(12,10))
sns.heatmap(numeric_features.corr(), square=True, vmax=0.8, cmap='mako')

The heatmap is the best way to get a quick overview of correlated features thanks to seaborn!

The 'availability' columns clearly move together, and so do the size columns ('accommodates', 'bedrooms', 'beds'). These variables give (almost) the same information, and hence it's a case of multicollinearity. In ML problems related to feature selection these heatmaps provide an excellent exploratory tool.

In [ ]:
# zoom in on just the strongly related columns
plt.figure(figsize=(12,12))
corr = numeric_features.corr()

# Keep only columns that are strongly correlated (>0.8) with at least one other column
# (every column counts itself once - its self-correlation is 1.0 - so "> 1"
#  means "correlated with at least one column besides itself")
high_corr_columns = corr.columns[(corr > 0.8).sum() > 1]

# annot=True prints the correlation value inside each square
sns.heatmap(corr.loc[high_corr_columns, high_corr_columns],
            square=True, center=0, annot=True, cmap='mako')

- 'accommodates', 'bedrooms' and 'beds' are strongly correlated. How many people a listing sleeps is a consequence of how many bedrooms and beds it has — they are like twin brothers (triplets, really). So it is hard to distinguish between them, and we would need only one of these in our analysis.
- 'availability_30', 'availability_60' and 'availability_90' are twins by construction: a night that is free in the next 30 days is also free in the next 60 and 90.
- 'estimated_occupancy_l365d', 'number_of_reviews_ltm' and 'reviews_per_month' move together — the occupancy estimate is calculated from recent reviews, so they are related by definition.
- The review sub-scores ('accuracy', 'cleanliness', 'value') all track the overall 'review_scores_rating': a guest who liked the stay tends to score everything highly.
- 'host_listings_count' and 'calculated_host_listings_count' are two ways of counting the same thing.

<h3>Pair Plot</h3>
Pair Plots between 'price' and interesting variables <br />
Visualisation of 'accommodates','bedrooms','beds','estimated_revenue_l365d','review_scores_rating','hosts_time_as_host_years','number_of_reviews' features with respect to price in the form of pair plot & scatter pair plot to create better understanding.

In [ ]:
# pair plot: every chosen column plotted against every other
# scatter charts off the diagonal show relationships between pairs;
# the smooth curves on the diagonal show each column's own distribution
# (patience - this one draws over a million points, so it takes a little while)
sns.set_theme()
columns = ['price','accommodates','bedrooms','beds','estimated_revenue_l365d','review_scores_rating','hosts_time_as_host_years','number_of_reviews']
sns.pairplot(df[columns], height=2, kind='scatter', diag_kind='kde')

<h1>What did we find?</h1>

Reading the price row: size is what moves it — 'accommodates' and 'bedrooms' (≈ 0.34 each) are the strongest links, with estimated revenue close behind. Just as interesting is what does <i>not</i> move it: 'review_scores_rating' correlates with price at only ≈ 0.07 — guests rate cheap stays about as highly as expensive ones. We also found multicollinearity in five separate families — the size trio, the availability windows, the review-activity columns, the review sub-scores, and the two host-listing counts — so a model would need only one column from each family. And several columns have missing values that would need treatment before modelling: listings with no reviews yet have no scores (nothing sensible to impute — the feature does not exist yet), while 'bathrooms' and 'beds' are simply unrecorded for some listings (candidates for imputation).